In [1]:
# Kết nối với Ollama server
from langchain.llms import Ollama

# Tạo chain LLM
from langchain.chains import LLMChain

# Tạo chain SQL query
from langchain.chains import create_sql_query_chain

# Tạo prompt template
from langchain.prompts import PromptTemplate

# Tạo tool để query database
from langchain.tools import QuerySQLDataBaseTool

# Tạo database
from langchain.sql_database import SQLDatabase

# Output
from langchain_core.output_parsers import StrOutputParser

# Runnable đa dạng các đối số
from langchain_core.runnables import RunnablePassthrough

# Operator lấy item
from operator import itemgetter

# Vector store
# from langchain_chroma import Chroma

# Embedding
# from langchain_openai import OpenAIEmbeddings

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cache
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache


In [2]:
from langchain_community.vectorstores import FAISS
from langchain.llms.ollama import Ollama

In [3]:
# Run sql file
import sqlite3
import os

# Define paths
sql_file = "../db/Chinook_Sqlite.sql"
db_path = "../db/chinook.sqlite3"

# Create db directory if it doesn't exist
os.makedirs(os.path.dirname(db_path), exist_ok=True)

try:
    # Create connection
    conn = sqlite3.connect(db_path)
    
    # Read and execute SQL file
    with open(sql_file, 'r', encoding='utf-8') as f:
        sql_script = f.read()
        conn.executescript(sql_script)
        
    print("Database created successfully!")
    
except Exception as e:
    print(f"Error creating database: {str(e)}")
    
finally:
    # Close connection
    if conn:
        conn.close()


Error creating database: [Errno 2] No such file or directory: '../db/Chinook_Sqlite.sql'


In [4]:
# Kết nối database với cài đặt tối ưu
# Chỉ lấy cấu trúc bảng mà không cần lấy mẫu dữ liệu từ các bảng (sample_rows_in_table_info=0) để giảm tải hệ thống.
db_path = "./ex1.sqlite3"
db = SQLDatabase.from_uri(
    f"sqlite:///{db_path}", 
    # sample_rows_in_table_info=0
)

# print(db.get_table_names())

def see_full_data(table_name):
    return db.run(f"SELECT * FROM {table_name}")

def get_all_info():
    return db.get_table_info(db.get_table_names())

get_all_info_tables = get_all_info()


C:\Users\MSI VN\AppData\Local\Temp\ipykernel_11460\3551784004.py:15: LangChainDeprecationWarning: The method `SQLDatabase.get_table_names` was deprecated in langchain-community 0.0.1 and will be removed in 1.0. Use get_usable_table_names instead.
  return db.get_table_info(db.get_table_names())


In [5]:
print(db.get_usable_table_names())

['Album', 'Artist', 'Track']


In [6]:
print(db.run("select * from album;")) # List[tuple]

[(1, 'Led Zeppelin IV', 1), (2, 'Physical Graffiti', 1), (3, 'Back in Black', 2), (4, 'Highway to Hell', 2), (5, 'The Dark Side of the Moon', 3), (6, 'The Wall', 3), (7, 'Abbey Road', 4), (8, 'Let It Be', 4), (9, 'A Night at the Opera', 5), (10, 'News of the World', 5), (11, 'Led Zeppelin IV', 1), (12, 'Alex', 1), (13, "Alex's Album", 1), (14, 'Alex Greatest Hits', 6)]


In [7]:
!ollama list

NAME                      ID              SIZE      MODIFIED    
qwen3-coder:480b-cloud    e30e45586389    -         5 days ago     
gemma3:latest             a2af6cc3eb7f    3.3 GB    7 days ago     
qwen3-vl:2b               0635d9d857d4    1.9 GB    7 days ago     
gpt-oss:120b-cloud        569662207105    -         9 days ago     
qwen3:1.7b                8f68893c685c    1.4 GB    10 days ago    


In [8]:


# Cấu hình bộ nhớ đệm (cache) để tăng tốc độ xử lý và giảm tải cho LLM
# Sử dụng InMemoryCache giúp lưu trữ kết quả tạm thời trong RAM, phù hợp với các truy vấn lặp lại hoặc tương tự.
set_llm_cache(InMemoryCache())

# Tạo đối tượng LLM Ollama với các cài đặt tối ưu
llm = Ollama(
    model="qwen3-coder:480b-cloud",  # Chọn mô hình phù hợp với bài toán
    temperature=0.1,            # Nhiệt độ thấp giúp tăng tính chính xác của kết quả
    num_ctx=1000,               # Tăng độ dài ngữ cảnh cho đầu vào để bao quát nhiều thông tin hơn
    num_thread=2,               # Tận dụng 2 luồng CPU để tăng tốc độ suy luận
    repeat_penalty=1.1,         # Phạt khi lặp lại, giảm hiện tượng lặp từ không cần thiết
    top_k=40,                   # Giới hạn chọn trong top 40 token, tăng tính chính xác
    top_p=0.9,                  # Áp dụng nucleus sampling để kiểm soát xác suất token
)

# Tạo chuỗi truy vấn cơ sở dữ liệu (query chain) với LLM đã được cache
# `k=2` cho phép truy vấn nhiều bảng trong cơ sở dữ liệu, mở rộng phạm vi truy vấn.
query_chain = create_sql_query_chain(llm, db, k=2)

# Khởi tạo công cụ thực thi truy vấn với cache, giảm số lần gọi truy vấn không cần thiết
query_tool = QuerySQLDataBaseTool(db=db)

# Tạo Prompt Template để định nghĩa câu trả lời đơn giản và ngắn gọn
# Template này kết hợp câu hỏi, truy vấn SQL, và kết quả truy vấn để sinh câu trả lời từ LLM.
answer_prompt = PromptTemplate.from_template(
    """    Dựa trên câu hỏi, truy vấn SQL và kết quả dưới đây, hãy trả lời câu hỏi của người dùng.

    Câu hỏi: {question}
    Truy vấn SQL: {query}
    Kết quả SQL: {result}
    
    Trả lời: """
)

In [9]:
template_get_sql_from_question = PromptTemplate.from_template(
'''
Context DB:
__
{tables_data}
__
Genertate sql query from question below:
__
{question}
__
Answer Format:
sql:result
'''
)

In [10]:
chain_sql_from_question = template_get_sql_from_question | llm | StrOutputParser()

In [11]:
get_all_tables_data = ",".join(db.get_usable_table_names())
get_all_tables_data

'Album,Artist,Track'

In [16]:
print(db.get_table_info(db.get_usable_table_names()))


CREATE TABLE "Album" (
	"AlbumId" BIGINT, 
	"Title" TEXT, 
	"ArtistId" BIGINT
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	Led Zeppelin IV	1
2	Physical Graffiti	1
3	Back in Black	2
*/


CREATE TABLE "Artist" (
	"ArtistId" BIGINT, 
	"Name" TEXT
)

/*
3 rows from Artist table:
ArtistId	Name
1	Led Zeppelin
2	AC/DC
3	Pink Floyd
*/


CREATE TABLE "Track" (
	"TrackId" BIGINT, 
	"Name" TEXT, 
	"AlbumId" BIGINT, 
	"Milliseconds" BIGINT, 
	"UnitPrice" FLOAT
)

/*
3 rows from Track table:
TrackId	Name	AlbumId	Milliseconds	UnitPrice
1	Stairway to Heaven	1	483000	0.99
2	Black Dog	1	295000	0.99
3	Kashmir	2	508000	0.99
*/


In [ ]:
sql_result = ""
for word in chain_sql_from_question.stream({
    "tables_data": f"My table in Database: {get_all_info()}",
    "question": "Led Zeppelin có bao nhiêu album"
}):
    sql_result += word
    print(word, end="")

```sql
SELECT COUNT(*) AS album_count
FROM Album a
JOIN Artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'Led Zeppelin';
```

In [18]:
clean_sql_md = sql_result.replace("```","").replace("sql","")
clean_sql_md

"\nSELECT COUNT(*) AS album_count\nFROM Album a\nJOIN Artist ar ON a.ArtistId = ar.ArtistId\nWHERE ar.Name = 'Led Zeppelin';\n"

In [19]:
db.run(clean_sql_md)

'[(2,)]'

In [20]:
'''
Từ 1 câu hỏi mà có thể truy vấn để đọc dữ liệu
'''
runnable_read_question = RunnablePassthrough.assign( # Merge the Dict input with the output produced by the mapping argument.
    # Đọc dữ liệu từ template, db, yêu cầu câu hỏi -> llm -> sinh ra câu truy vấn
    # x = {}
    sql_result = lambda x: (template_get_sql_from_question | llm | StrOutputParser())
).assign(
    # x = {"sql_result": value}
    # Clean sql markdown
    clean_sql = lambda x: x["sql_result"].replace("```","").replace("sql","")
).assign(
    # x = {"sql_result":value, "clean_sql":value}
    db_result = lambda x: db.run(x["clean_sql"])
)

In [64]:
template_insert_data = PromptTemplate.from_template(
'''
Context DB:
__
{tables_data}
__
Genertate sql query to insert data for database from question below:
__
{question}
__
Rule:
- Không tự ý chọn ID, mà phải dùng function để tính toán lại giá trị id đang có
- Dữ liệu database khi insert có thể tự nghĩ ra không cần người nhập cung cấp
__
Answer Format:
sql:result
'''
)

In [58]:
'''
Từ 1 câu hỏi mà có thể truy vấn để đọc dữ liệu
'''
runnable_read_question = RunnablePassthrough.assign( # Merge the Dict input with the output produced by the mapping argument.
    # Đọc dữ liệu từ template, db, yêu cầu câu hỏi -> llm -> sinh ra câu truy vấn
    # x = {}
    sql_result = lambda x: (template_insert_data | llm | StrOutputParser())
).assign(
    # x = {"sql_result": value}
    # Clean sql markdown
    clean_sql = lambda x: x["sql_result"].replace("```","").replace("sql:","")
).assign(
    # x = {"sql_result":value, "clean_sql":value}
    db_result = lambda x: db.run(x["clean_sql"])
)

In [ ]:
'''
Lỗi: khi thêm thì insert từng lần một do cái thư viện db không execute once cùng lúc nhiều lệnh thì tách ra thành từng lệnh 1.
'''

In [62]:
dap_an = runnable_read_question.invoke({
    "tables_data": f"My table in Database: {get_all_info()}",
    "question": f'''thêm nghệ sĩ Alex'''
})

In [ ]:
dap_an = runnable_read_question.invoke({
    "tables_data": f"My table in Database: {get_all_info()}",
    "question": f'''thêm album cho nghệ sĩ có tên Alex'''
})

In [56]:
print(dap_an["clean_sql"])

INSERT INTO "Album" ("AlbumId", "Title", "ArtistId") VALUES ((SELECT COALESCE(MAX("AlbumId"), 0) + 1 FROM "Album"), 'Led Zeppelin IV', (SELECT "ArtistId" FROM "Artist" WHERE "Name" = 'Led Zeppelin'));


In [57]:
db.run(dap_an["clean_sql"])

''

In [22]:
dap_an = runnable_read_question.invoke({
    "tables_data": f"My table in Database: {get_all_info()}",
    "question": "Led Zeppelin có bao nhiêu album"
})

In [26]:
dap_an["db_result"]

'[(2,)]'

In [8]:
nl2query = (query_chain.invoke({"question": "How many employees are there?"}))


In [60]:
# Tạo prompt template để tạo truy vấn SQL với thông tin bảng
smart_nl2query = PromptTemplate.from_template(
    """
    Dựa vào dữ liệu bảng đã có:\n
    {get_all_info_tables}
    Hãy tạo truy vấn SQL cho câu hỏi: {question}, từ dữ liệu bảng đã có. Kiểm tra lại truy vấn SQL để đảm bảo đúng với bảng đã có.
    """
)

# Tạo chain để xử lý câu hỏi
try:
    # Tạo pipeline xử lý:
    # 1. Tạo truy vấn SQL từ smart_nl2query template
    # 2. Xử lý bởi LLM để sinh truy vấn SQL
    # 3. Parse kết quả thành string
    result = (
smart_nl2query
        | llm 
        | query_chain
        | StrOutputParser()
    )
    
    # Thực thi chain
    smart_nl2query_invoke = result.invoke({"question": "How many employee are there?"})
    
    # In kết quả
    print("Kết quả truy vấn:")
    print(smart_nl2query_invoke)

except Exception as e:
    print(f"Lỗi khi thực thi chain: {str(e)}")


Lỗi khi thực thi chain: "Input to PromptTemplate is missing variables {'result'}.  Expected: ['query', 'question', 'result'] Received: ['question', 'result_nl2query', 'query']\nNote: if you intended {result} to be part of the string and not a variable, please escape it with double curly braces like: '{{result}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT"


In [14]:
print(nl2query.split("```sql")[1].split("```")[0].strip())
# Lấy query từ kết quả trả về
query = nl2query.split("```sql")[1].split("```")[0].strip()
result = query_tool.run(query)
print(db.get_table_names())
print(result)


SELECT COUNT(*) FROM Employees;
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Error: (sqlite3.OperationalError) no such table: Employees
[SQL: SELECT COUNT(*) FROM Employees;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


C:\Users\Admin\AppData\Local\Temp\ipykernel_8924\2358363832.py:5: LangChainDeprecationWarning: The method `SQLDatabase.get_table_names` was deprecated in langchain-community 0.0.1 and will be removed in 1.0. Use :meth:`~get_usable_table_names` instead.
  print(db.get_table_names())


In [ ]:
# Tạo agent tối ưu với nhiều bước xử lý trong một pipeline
# - Sử dụng `RunnablePassthrough` để tạo chuỗi pipeline có các bước tuần tự: 
#    1. Nhận câu hỏi và tạo truy vấn với `query_chain`
#    2. Thực thi truy vấn với `query_tool`
#    3. Tạo câu trả lời từ prompt và gửi đến LLM để tạo output cuối cùng.
agent = (
    RunnablePassthrough.assign(
        query=query_chain,
    ).assign(
        result=itemgetter("query") | query_tool,
    ).assign(
        table_name=lambda x: x["query"].lower().split("from")[1].strip().split()[0] if "from" in x["query"].lower() else None
    ).assign(
        table_data=lambda x: see_full_data(x["table_name"]) if x["table_name"] is not None else None
    ).assign(
        final_result=lambda x: x["result"] if x["table_data"] is None else x["table_data"]
    )
    | answer_prompt
    | llm
    | StrOutputParser()
).with_config({
    "run_name": "sql_qa",
    "max_concurrency": 2,
    "cache": True
})

# Kiểm tra agent với batch processing để xử lý nhiều câu hỏi đồng thời
# Batch processing tăng hiệu suất cho các truy vấn lặp lại.
batch_questions = [{"question": "How many employees are there?"}]

# In ra query và kết quả cho từng câu hỏi
for question in batch_questions:
    # Lấy query từ query chain
    query = query_chain.invoke(question)
    print(f"\nQuestion: {question['question']}")
    print(f"Generated SQL Query: {query}")
    
    # Thực thi query và lấy kết quả
    result = query_tool.run(query)
    print(f"Query Result: {result}")
    
    # Lấy câu trả lời cuối cùng
    response = agent.invoke(question)
    print(f"Final Response: {response}")